# Text-JEPA v4 · content-aware SIGReg, no EMA

**Задача:** предотвратить позиционный shortcut, который наблюдался в v3 на 20M токенов. В v3 token-wise SIGReg считался по объединённым позициям; это позволяло получить разнообразное распределение за счёт позиций без различий между текстами.

## Изменения v4

1. **SIGReg условно на позиции**: для каждого из 128 скрытых положений вычисляется Epps–Pulley статистика **по разным документам**, не по токенам разных позиций. Решение z(x,t)=p(t) теперь резко штрафуется.
2. **64 независимых документа на шаг**, по одному 512-токеновому фрагменту из документа: 8 microbatches × 8 документов. Вариант градиентного кеширования делает **два forward**: без графа для построения общего loss и затем с графом по одному microbatch для переноса градиентов на параметры. Это аналог GradCache, **не** EMA и **не** stop-gradient для целевой ветви; target обновляется обратным распространением.
3. **Контроль исходного / shuffled / constant-token контекста** с retrieval@1 **по 64 документам на одинаковой скрытой позиции** (chance 1/64=1.5625%).
4. Логируются доля содержательной дисперсии, effective rank, loss компонентов, скорости токенов, и есть ранняя остановка, если вариативность по текстам исчезла.
5. Корпус **FineWeb-Edu sample-10BT**, Transformer 12×768 (≈125M), 512 токенов, 25% span, BF16, 20M токенов для пилота. При подтверждении качества увеличить до 500M / 2B.

**Принципиально:** это исследовательская Text-JEPA с условной SIGReg, не каноническая LeJEPA. Применение SIGReg к batch из B=64 при каждом фиксированном t более корректно для этой цели, но **не гарантирует семантическое понимание**. Модель может предсказывать другие поверхностные текстовые признаки. Оценивайте также downstream-пробы на семантику.

Исходный SIGReg/Epps–Pulley: https://github.com/galilai-group/lejepa/blob/main/MINIMAL.md

**Ресурсы:** градиентное кеширование делает 2 forward + 1 backward вместо 1 forward + 1 backward, снижая пиковое хранение графа, но не ускоряя обучение. Пока не проверено на GB10/DGX Spark; сначала выполнить **smoke test 2 шага**. Точный курсор стрима не сохраняется при resume.

In [ ]:
import os, math, random, time, json, copy, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch",torch.__version__,"device",DEVICE)
if DEVICE.type=="cuda":
    print(torch.cuda.get_device_name(0))
    torch.set_float32_matmul_precision("high")

## 1. Параметры

В отличие от v3, 8 микробатчей агрегируются в одну статистическую выборку из 64 **разных документов**. Это требует двух forward на каждом шаге, зато статистика SIGReg при заданной позиции формируется по B=64 независимым текстам. Для первой проверки можно установить token_budget=2*32768. Не изменяйте данные одновременно с архитектурой.

In [ ]:
@dataclass
class Config:
    seed:int=42
    preset:str="base"            # "large" = 16x1024, если ресурсы позволяют
    seq_len:int=512
    span_frac:float=0.25
    proj_dim:int=256
    micro_batch:int=8
    grad_accum:int=8              # SIGReg вычисляется по ОБЩЕМУ батчу 8*8=64
    token_budget:int=20_000_000   # smoke test: 2*8*8*512; 500M после подтверждения
    lr:float=2e-4
    weight_decay:float=0.05
    sigreg_weight:float=0.05
    sigreg_directions:int=64
    sigreg_knots:int=17
    dropout:float=0.0             # НЕ менять: два forward должны быть детерминированы
    max_grad_norm:float=1.0
    eval_every:int=25
    save_every:int=100
    val_documents:int=64
    data_buffer:int=2048
    max_doc_tokens:int=4096
    out_dir:str="runs/text_span_sigreg_v4"
    hf_repo:str="HuggingFaceFW/fineweb-edu"
    hf_config:str="sample-10BT"
    tokenizer:str="gpt2"
cfg=Config()
if cfg.preset=="base":
    depth,width,heads=12,768,12
elif cfg.preset=="large":
    depth,width,heads=16,1024,16
else:
    raise ValueError("preset must be base or large")
assert cfg.dropout==0.0, "GradCache replay requires dropout=0"
assert 0<cfg.span_frac<0.5 and cfg.seq_len>=64
assert cfg.grad_accum>0 and cfg.micro_batch>=2
assert cfg.val_documents>=2 and cfg.val_documents%cfg.micro_batch==0
assert 0<cfg.sigreg_weight<1
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(cfg.seed)
global_batch=cfg.micro_batch*cfg.grad_accum
tokens_per_step=global_batch*cfg.seq_len
max_steps=math.ceil(cfg.token_budget/tokens_per_step)
print(f"model: {cfg.preset}, layers={depth}, width={width}, heads={heads}")
print(f"global documents/step={global_batch}, tokens/step={tokens_per_step:,}, steps={max_steps}")
print(f"SIGReg uses B={global_batch} different documents at each FIXED position")

## 2. Потоковые данные с одним фрагментом из документа

Для корректности условной SIGReg по документам: берём **не более одного блока из каждого документа** и фиксируем train/val разбиение по SHA256(\`id\`), примерно 99/1. Выбор окна внутри документа детерминирован по хешу ID. Короткие документы (<512 токенов) пропускаются. У валидатора фиксирован список 64 документов.

Это гарантирует отсутствие повторов одного ID в пределах одного шага при условии отсутствия дубликатов ID в корпусе. **Не гарантирует семантическую дедупликацию** одинакового текста из разных источников. При resume итератор начинается заново и повторяет пройденные документы; для масштабного претрейна нужно сохранять курсор стрима.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(cfg.tokenizer)
def doc_hash(ex):
    identifier=str(ex.get("id") or ex.get("text","")[:1000])
    return int.from_bytes(hashlib.sha256(identifier.encode("utf-8")).digest()[:8],"big")
def is_validation(ex):
    return (doc_hash(ex)%100)==0
class FineWebDocuments(IterableDataset):
    def __init__(self,part,shuffle=False):
        assert part in ("train","val")
        self.part=part; self.shuffle=shuffle
    def __iter__(self):
        ds=load_dataset(cfg.hf_repo,cfg.hf_config,split="train",streaming=True)
        if self.shuffle:
            ds=ds.shuffle(seed=cfg.seed,buffer_size=cfg.data_buffer)
        for ex in ds:
            if is_validation(ex)!=(self.part=="val"):continue
            content=str(ex.get("text",""))
            if len(content)<cfg.seq_len:continue
            ids=tokenizer(content,add_special_tokens=False,truncation=True,
                          max_length=cfg.max_doc_tokens)["input_ids"]
            if len(ids)<cfg.seq_len:continue
            offset=doc_hash(ex)%(len(ids)-cfg.seq_len+1)
            yield torch.tensor(ids[offset:offset+cfg.seq_len],dtype=torch.long)

train_loader=DataLoader(FineWebDocuments("train",True),
    batch_size=cfg.micro_batch,drop_last=True,num_workers=0,pin_memory=DEVICE.type=="cuda")
validation_dataset=FineWebDocuments("val",False)
validation_loader=DataLoader(validation_dataset,
    batch_size=cfg.micro_batch,drop_last=True,num_workers=0,pin_memory=DEVICE.type=="cuda")
val_chunks=[]
for i,batch in enumerate(validation_loader):
    if i>=cfg.val_documents//cfg.micro_batch:break
    val_chunks.append(batch)
assert len(val_chunks)==cfg.val_documents//cfg.micro_batch, "Not enough validation documents"
val_x=torch.cat(val_chunks,dim=0)
assert val_x.shape==(cfg.val_documents,cfg.seq_len)
print(f"Cached {len(val_x)} validation documents, {len(val_chunks)} microbatches")
print("Example:",tokenizer.decode(val_x[0,:50]))

## 3. Маскирование

В пределах **всего общего батча** выбирается один случайный start для 128-токенового скрытого span. У всех B=64 документов скрытые позиции идентичны — регуляризация и retrieval сравнивают тексты при фиксированном t. На разных optimizer шагах start меняется. На валидации — строго центрированный span.

In [ ]:
def split_span(x,start):
    b,l=x.shape
    span=max(2,round(l*cfg.span_frac))
    assert 0<=start<=l-span
    pos=torch.arange(l,device=x.device)[None,:].expand(b,-1)
    mask=(pos>=start)&(pos<start+span)
    visible_pos=pos[~mask].reshape(b,l-span)
    target_pos=pos[mask].reshape(b,span)
    return (x.gather(1,visible_pos),visible_pos,
            x.gather(1,target_pos),target_pos)

def sample_span_start():
    span=max(2,round(cfg.seq_len*cfg.span_frac))
    return random.randint(0,cfg.seq_len-span)

## 4. Shared encoder + 128 токеновых прогнозов

Веса энкодера и projection обучаются в обеих ветвях, **EMA отсутствует**. Используется LayerNorm без affine на выходе projection, но он не защищает от *позиционного* shortcut. Поэтому регуляризация должна видеть различия по документам, а не только по позициям.

In [ ]:
class Backbone(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.word=nn.Embedding(vocab,width)
        self.position=nn.Embedding(cfg.seq_len,width)
        layer=nn.TransformerEncoderLayer(width,heads,4*width,dropout=cfg.dropout,
            activation="gelu",batch_first=True,norm_first=True)
        self.layers=nn.TransformerEncoder(layer,depth,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(width)
    def forward(self,ids,positions):
        h=self.word(ids)+self.position(positions)
        return self.norm(self.layers(h))
class TokenSpanHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.target_project=nn.Sequential(nn.LayerNorm(width),
            nn.Linear(width,cfg.proj_dim),
            nn.LayerNorm(cfg.proj_dim,elementwise_affine=False))
        self.query=nn.Embedding(cfg.seq_len,width)
        self.cross=nn.MultiheadAttention(width,heads,batch_first=True)
        self.predictor=nn.Sequential(nn.LayerNorm(width),nn.Linear(width,width),
            nn.GELU(),nn.Linear(width,cfg.proj_dim),
            nn.LayerNorm(cfg.proj_dim,elementwise_affine=False))
    def forward(self,context_h,target_h,tp):
        query=self.query(tp)
        att,_=self.cross(query,context_h,context_h,need_weights=False)
        pred=self.predictor(query+att)
        target=self.target_project(target_h)
        return pred,target

encoder=Backbone(len(tokenizer)).to(DEVICE)
head=TokenSpanHead().to(DEVICE)
params=list(encoder.parameters())+list(head.parameters())
print("Encoder parameters:",f"{sum(p.numel() for p in encoder.parameters())/1e6:.1f}M")
print("Total parameters:",f"{sum(p.numel() for p in params)/1e6:.1f}M")
@torch.no_grad()
def model_probe_size():
    return sum(p.numel() for p in params)
print("EMA=NONE; target gradients ENABLED")

def forward_views(x,start):
    cid,cp,tid,tp=split_span(x,start)
    hc=encoder(cid,cp)
    ht=encoder(tid,tp)
    return head(hc,ht,tp)

## 5. Position-conditional SIGReg (Epps–Pulley)

Для каждого скрытого индекса t отдельно берём 64 эмбеддинга (по одному из 64 документов).

\[
\mathcal L_{\mathrm{SIGReg},t}=\mathrm{EP}\big(\{z(x_i,t)\}_{i=1}^{64}\big), \quad
\mathcal L_{\mathrm{SIGReg}}=\tfrac1{128}\sum_t \mathcal L_{\mathrm{SIGReg},t}.
\]

Здесь используется схема интегрирования из LeJEPA \`MINIMAL.md\`: 17 узлов на [0,3], веса трапеций умножаются на Gaussian window, случайные единичные проекции, статистика умножается на B=64 **для каждой позиции**. Отдельно считаем штраф для predicted и target. При \`z(x,t)=p_t\` SIGReg на данной позиции становится большой, даже если представления разных позиций разнообразны.

Контрольный тест: SIGReg для константных эмбеддингов должен заметно превышать SIGReg для гауссовых векторов. Это не строгая гарантия полезности эмбеддингов; независимая семантическая оценка остаётся необходимой.

In [ ]:
class ConditionalSIGReg(nn.Module):
    def __init__(self,knots=17,directions=64):
        super().__init__()
        t=torch.linspace(0,3,knots,dtype=torch.float32)
        dt=3/(knots-1)
        w=torch.full((knots,),2*dt)
        w[0]=dt;w[-1]=dt
        gaussian=torch.exp(-0.5*t.square())
        self.register_buffer("times",t)
        self.register_buffer("phi",gaussian)
        self.register_buffer("weights",w*gaussian)
        self.directions=directions
    def forward(self,z):
        assert z.ndim==3,"Expected [documents, masked_positions, latent_dim]"
        b,t,d=z.shape
        assert b>=2
        z=z.float()
        # Same random projections for each fixed position.
        directions=F.normalize(torch.randn(d,self.directions,device=z.device),dim=0)
        projections=torch.einsum("btd,dk->tbk",z,directions)
        phase=projections.unsqueeze(-1)*self.times
        real=phase.cos().mean(dim=1)  # mean over DOCUMENTS, not positions
        imag=phase.sin().mean(dim=1)
        err=(real-self.phi).square()+imag.square()
        return (err@self.weights).mean()*b

sigreg=ConditionalSIGReg(cfg.sigreg_knots,cfg.sigreg_directions).to(DEVICE)
with torch.no_grad():
    n=64;t=16;d=cfg.proj_dim
    zero=torch.zeros(n,t,d,device=DEVICE)
    normal=torch.randn(n,t,d,device=DEVICE)
    penalty_zero=float(sigreg(zero))
    penalty_normal=float(sigreg(normal))
    print(f"SIGReg sanity: zero={penalty_zero:.4f}, gaussian={penalty_normal:.4f}")
    assert penalty_zero>5*penalty_normal, "SIGReg implementation sanity failed"

def objective(pred,target):
    # Each: [global_batch (64), masked span (128), 256]
    mse=F.mse_loss(pred,target)
    reg_p=sigreg(pred)
    reg_t=sigreg(target)
    reg=0.5*(reg_p+reg_t)
    loss=(1-cfg.sigreg_weight)*mse+cfg.sigreg_weight*reg
    return loss,mse,reg,reg_p,reg_t

opt=torch.optim.AdamW(params,lr=cfg.lr,weight_decay=cfg.weight_decay)

@torch.no_grad()
def evaluate():
    encoder.eval();head.eval()
    fixed_start=(cfg.seq_len-round(cfg.seq_len*cfg.span_frac))//2
    original=[];wrong=[];constant=[];targets=[]
    # Important: incorrect context belongs to ANOTHER document, target remains from original.
    wrong_x=val_x.roll(1,dims=0)
    eos=int(tokenizer.eos_token_id)
    for j in range(0,len(val_x),cfg.micro_batch):
        xb=val_x[j:j+cfg.micro_batch].to(DEVICE,non_blocking=True)
        xw=wrong_x[j:j+cfg.micro_batch].to(DEVICE,non_blocking=True)
        xc=torch.full_like(xb,eos)
        with torch.autocast(device_type=DEVICE.type,dtype=torch.bfloat16,
                            enabled=DEVICE.type=="cuda"):
            p,t=forward_views(xb,fixed_start)
            pw,_=forward_views(xw,fixed_start)
            pc,_=forward_views(xc,fixed_start)
        original.append(p.float());targets.append(t.float())
        wrong.append(pw.float());constant.append(pc.float())
    p=torch.cat(original,0);t=torch.cat(targets,0)
    pw=torch.cat(wrong,0);pc=torch.cat(constant,0)
    _,mse,_,_,_=objective(p,t)
    _,mse_wrong,_,_,_=objective(pw,t)
    _,mse_const,_,_,_=objective(pc,t)
    # Separate regularization diagnostics; sampling of random directions causes small jitter.
    r_pred=float(sigreg(p));r_target=float(sigreg(t))
    def retrieval(pred,truth):
        query=F.normalize(pred,dim=-1)
        candidates=F.normalize(truth,dim=-1)
        # [position, query_document, candidate_document].
        scores=torch.einsum("btd,ctd->tbc",query,candidates)
        b=len(query)
        return float((scores.argmax(-1)==torch.arange(b,device=query.device)[None,:]).float().mean())
    # Variance over independent documents at FIXED position, not variance across positions.
    variation_p=float(p.var(dim=0,unbiased=False).mean().sqrt())
    variation_t=float(t.var(dim=0,unbiased=False).mean().sqrt())
    total_t=float(t.reshape(-1,t.size(-1)).var(0,unbiased=False).mean())
    content_fraction=(variation_t**2)/(total_t+1e-12)
    # Across all tokens and documents; rank=256 possible for B=64 & T=128.
    centered=t.reshape(-1,t.size(-1))-t.reshape(-1,t.size(-1)).mean(0)
    eig=torch.linalg.eigvalsh(centered.T@centered/max(1,len(centered)-1)).clamp_min(0)
    weights=eig/eig.sum().clamp_min(1e-9)
    rank=torch.exp(-(weights*(weights+1e-12).log()).sum())
    return {
        "pred_loss":float(mse),
        "shuffled_mse":float(mse_wrong),
        "constant_mse":float(mse_const),
        "sigreg_pred":r_pred,"sigreg_target":r_target,
        "same_pos_at1":retrieval(p,t),
        "shuffled_at1":retrieval(pw,t),
        "constant_at1":retrieval(pc,t),
        "chance_at1":1/len(val_x),
        "pred_content_std":variation_p,
        "target_content_std":variation_t,
        "target_content_fraction":content_fraction,
        "rank":float(rank)
    }

print("Initial evaluation:",evaluate())

## 6. Обучение с точным глобальным loss через GradCache

### Почему используется два forward

Обычное \`grad_accum=8\` вычисляет SIGReg восемь раз по B=8 и **не даёт** статистику из B=64. Здесь реализовано двухпроходное накопление, которое даёт точный градиент общей loss (за исключением стандартных численных ошибок BF16):

1. Для 8 microbatches на текущих параметрах — \`torch.no_grad()\` forward. Сохраняем \`pred,target\` (не графы модели).
2. Конкатенируем 64 документов, вычисляем общую MSE и **position-conditional SIGReg** с автодифференцированием только по латентам.
3. Второй forward по каждому microbatch (тот же \`start\`, \`dropout=0\`), передаём градиенты латентов назад через энкодер и предиктор, суммируя градиенты параметров. Обновление оптимизатора происходит один раз.

На каждом шаге ~2 forward + 1 backward, больше времени, но заметно меньше peak memory, чем хранить граф всех 64 документов. При нехватке RAM уменьшайте \`micro_batch\` и увеличивайте \`grad_accum\`, сохранив global_batch>=64.

**Сначала smoke test:** установите \`cfg.token_budget=2*cfg.micro_batch*cfg.grad_accum*cfg.seq_len\` и перезапустите ядро. Только после этого 20M. Для v4 нужна свежая инициализация; **не возобновляйте v3 checkpoint**.

Resume: восстанавливаются веса и оптимизатор, но не cursor потока HF. Для длинных прогонов потребуется полноценная сохранённая позиция данных.

In [ ]:
Path(cfg.out_dir).mkdir(parents=True,exist_ok=True)
checkpoint_path=Path(cfg.out_dir)/"latest.pt"
resume=False  # True only for v4 checkpoint with matching config
start_step=0;history=[]
if resume:
    blob=torch.load(checkpoint_path,map_location="cpu",weights_only=False)
    for key in ("preset","seq_len","proj_dim","micro_batch","grad_accum","sigreg_weight","sigreg_directions"):
        assert blob["config"][key]==getattr(cfg,key),f"Resume config mismatch: {key}"
    encoder.load_state_dict(blob["encoder"])
    head.load_state_dict(blob["head"])
    opt.load_state_dict(blob["optimizer"])
    start_step=int(blob["step"]);history=list(blob["history"])
    print("Resuming v4 at",start_step)
def save_checkpoint(step):
    tmp=checkpoint_path.with_suffix(".tmp")
    torch.save({"config":asdict(cfg),"step":step,
        "encoder":encoder.state_dict(),"head":head.state_dict(),
        "optimizer":opt.state_dict(),"history":history},tmp)
    tmp.replace(checkpoint_path)
iterator=iter(train_loader)
t0=time.time()
collapse_streak=0
for step in tqdm(range(start_step+1,max_steps+1),desc="Text-JEPA v4 position-aware SIGReg"):
    encoder.train();head.train()
    opt.zero_grad(set_to_none=True)
    warmup=max(10,round(max_steps*.03))
    if step<=warmup:factor=max(.01,step/warmup)
    else:
        portion=(step-warmup)/max(1,max_steps-warmup)
        factor=.1+.9*.5*(1+math.cos(math.pi*portion))
    for group in opt.param_groups:group["lr"]=cfg.lr*factor
    start=sample_span_start()  # same hidden positions for all 64 docs
    micro_cpu=[]
    for m in range(cfg.grad_accum):
        try:micro_cpu.append(next(iterator))
        except StopIteration:
            iterator=iter(train_loader)
            micro_cpu.append(next(iterator))
    cached_pred=[];cached_target=[]
    # Pass A: collect latent values without storing model autograd graphs.
    with torch.no_grad():
        for xb in micro_cpu:
            x=xb.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,dtype=torch.bfloat16,
                                enabled=DEVICE.type=="cuda"):
                p,t=forward_views(x,start)
            cached_pred.append(p.float().detach())
            cached_target.append(t.float().detach())
    global_pred=torch.cat(cached_pred,0).requires_grad_(True)
    global_target=torch.cat(cached_target,0).requires_grad_(True)
    loss,mse,reg,regp,regt=objective(global_pred,global_target)
    if not torch.isfinite(loss):
        save_checkpoint(step-1)
        raise RuntimeError("Non-finite objective; checkpoint saved")
    loss.backward()
    gp=global_pred.grad.detach()
    gt=global_target.grad.detach()
    # Pass B: replay the same deterministic forward with autograd; inject cached grads.
    # No optimizer.step() until ALL 8 microbatches are replayed.
    for m,xb in enumerate(micro_cpu):
        x=xb.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,dtype=torch.bfloat16,
                            enabled=DEVICE.type=="cuda"):
            p,t=forward_views(x,start)
        sl=slice(m*cfg.micro_batch,(m+1)*cfg.micro_batch)
        torch.autograd.backward([p.float(),t.float()],
                                [gp[sl].contiguous(),gt[sl].contiguous()])
    gn=float(nn.utils.clip_grad_norm_(params,cfg.max_grad_norm))
    opt.step()
    if step==1 or step%cfg.eval_every==0 or step==max_steps:
        result={"step":step,"tokens_seen":step*tokens_per_step,
            "train_pred":float(mse.detach()),"train_sigreg":float(reg.detach()),
            "train_sigreg_pred":float(regp.detach()),"train_sigreg_target":float(regt.detach()),
            "grad_norm":gn,"lr":opt.param_groups[0]["lr"],
            "seconds":time.time()-t0,"tokens_per_sec":step*tokens_per_step/max(time.time()-t0,1e-3)}
        em=evaluate()
        result.update({"val_"+k:v for k,v in em.items()})
        history.append(result)
        print(result)
        # Do not confuse per-vector LayerNorm std with variation among DOCUMENTS.
        collapsed=(em["target_content_std"]<.1 or em["pred_content_std"]<.1)
        collapse_streak=(collapse_streak+1) if collapsed else 0
        if collapse_streak>=2:
            save_checkpoint(step)
            print("COLLAPSE WARNING: cross-document variance too low. Stopped early.")
            break
    if step%cfg.save_every==0 or step==max_steps:
        save_checkpoint(step)
if history:
    save_checkpoint(history[-1]["step"])
print("Finished step",history[-1]["step"] if history else start_step)
print("Checkpoint",checkpoint_path)

## 7. Отчёт

Успех v4 требует одновременно:
- \`val_target_content_std\` и \`val_pred_content_std\` **не стремятся к нулю**; \`val_target_content_fraction\` показывает долю дисперсии между документами при фиксированной позиции.
- \`val_same_pos_at1\` **выше 1/64 = 1.5625%** и явно выше \`val_shuffled_at1\` / \`val_constant_at1\`.
- \`val_pred_loss\` снижается без ухудшения диагностик распределения.
- Лучший внешний критерий — separate semantic linear probe; retrieval только промежуточная метрика.

Публикуется \`config.json\`, \`metrics.json\`, \`training.png\`; модельные веса и тексты остаются локально.

In [ ]:
def export_metrics():
    out=Path(cfg.out_dir);out.mkdir(parents=True,exist_ok=True)
    (out/"config.json").write_text(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
    (out/"metrics.json").write_text(json.dumps({
        "schema_version":4,
        "method":"Text-JEPA shared encoder, no EMA, position-conditional SIGReg, global GradCache",
        "tokens_seen":history[-1]["tokens_seen"] if history else 0,
        "n_documents_per_update":global_batch,
        "note":"One span per document, one common mask start per batch. HF stream cursor is NOT saved.",
        "validation_note":"64-doc same-position retrieval, chance=1/64; shuffled/constant controls",
        "history":history},indent=2,ensure_ascii=False))
    fig,axes=plt.subplots(2,3,figsize=(14,8))
    plots=["val_pred_loss","val_sigreg_target","val_target_content_std",
           "val_pred_content_std","val_same_pos_at1","val_rank"]
    for ax,key in zip(axes.flat,plots):
        ax.plot([h["tokens_seen"]/1e6 for h in history],[h[key] for h in history],marker=".")
        ax.set_title(key);ax.set_xlabel("Seen tokens (M)");ax.grid(alpha=.25)
        if key=="val_same_pos_at1" and history:
            ax.plot([h["tokens_seen"]/1e6 for h in history],
                    [h["val_shuffled_at1"] for h in history],label="shuffled",linestyle="--")
            ax.axhline(1/cfg.val_documents,linestyle=":",label="random")
            ax.legend()
    fig.tight_layout();fig.savefig(out/"training.png",dpi=140)
    plt.show();plt.close(fig)
    print("Exported report",out)
    return out
report_dir=export_metrics()

## 8. Публикация в ветку GitHub

Результаты публикуются строго в отдельную директорию \`results/text_span_sigreg_v4/<timestamp>_base_42/\`, никаких checkpoint, ключей, учебных текстов или ноутбуков не отправляется. Настройка git name/email/SSH та же, что в v3. Для автоматической отправки вызовите \`publish_v4_report()\` после обучения.

In [ ]:
import subprocess
from datetime import datetime,timezone
EXPECTED_BRANCH="experiment/text-i-jepa-notebook-20261113"
def run_git(*args,root=None,timeout=90):
    p=subprocess.run(["git",*args],cwd=root,stdout=subprocess.PIPE,
                     stderr=subprocess.PIPE,text=True,timeout=timeout)
    if p.returncode:
        raise RuntimeError("git "+" ".join(args)+" failed: "+p.stderr[-1000:])
    return p.stdout.strip()
def publish_v4_report():
    if not history:raise RuntimeError("Train first")
    root=Path(run_git("rev-parse","--show-toplevel")).resolve()
    if run_git("branch","--show-current",root=root)!=EXPECTED_BRANCH:
        raise RuntimeError("Switch to experiment branch")
    origin=run_git("remote","get-url","origin",root=root)
    if origin.rstrip("/").removesuffix(".git") not in (
        "https://github.com/crystalcali/salm",
        "git@github.com:crystalcali/salm",
        "ssh://git@github.com/crystalcali/salm"):
        raise RuntimeError("Unexpected origin "+origin)
    if run_git("diff","--cached","--name-only",root=root):
        raise RuntimeError("Other staged files found; commit or unstage first")
    source=export_metrics()
    timestamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    rel=Path("results")/"text_span_sigreg_v4"/(timestamp+"_"+cfg.preset+"_"+str(cfg.seed))
    dest=root/rel;dest.mkdir(parents=True,exist_ok=False)
    names=("config.json","metrics.json","training.png")
    for name in names:(dest/name).write_bytes((source/name).read_bytes())
    files=[str(rel/name) for name in names]
    run_git("add","--",*files,root=root)
    if set(run_git("diff","--cached","--name-only",root=root).splitlines())!=set(files):
        run_git("reset","--",*files,root=root)
        raise RuntimeError("Unexpected git index paths")
    run_git("commit","-m",f"results: text JEPA v4 {timestamp}",root=root)
    run_git("push","origin",f"HEAD:refs/heads/{EXPECTED_BRANCH}",root=root,timeout=120)
    url="https://github.com/crystalcali/salm/tree/"+EXPECTED_BRANCH+"/"+rel.as_posix()
    print("Published:",url)
    return url
# Once finished: report_url=publish_v4_report()